In [0]:
# Databricks notebook source

# ============================================
# PARAMETRO DE ENV VINDO DA PIPELINE / JOB
# ============================================
dbutils.widgets.text("env", "")
p_env = dbutils.widgets.get("env")

env_norm = p_env.strip().lower()
if env_norm in ("hmg", "homol", "hml"):
    env_suffix = "hml"
elif env_norm in ("prod", "prd"):
    env_suffix = "prd"
else:
    env_suffix = "dev"

control_catalog = f"platform_{env_suffix}"

spark.sql(f"""
CREATE OR REPLACE PROCEDURE {control_catalog}.utils.proc_prepare_target
(
  IN p_config_name STRING,
  IN p_layer       STRING,
  IN p_system      STRING,
  IN p_env         STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN
  DECLARE v_target_catalog STRING;
  DECLARE v_target_schema STRING;
  DECLARE v_target_object STRING;
  DECLARE v_target_format STRING;
  DECLARE v_domain STRING;
  DECLARE v_env STRING;
  DECLARE v_target_location STRING;
  DECLARE v_schema_ddl STRING;
  DECLARE v_create_schema STRING;
  DECLARE v_create_table STRING;
  DECLARE v_target_exists INT;

  -- load config
  CALL {control_catalog}.utils.proc_load_config(
    p_config_name,
    p_layer,
    p_system
  );

  -- normalize env
  SET v_env = CASE
    WHEN LOWER(TRIM(p_env)) IN ('hmg', 'homol', 'hml') THEN 'hml'
    WHEN LOWER(TRIM(p_env)) IN ('prod', 'prd') THEN 'prd'
    ELSE LOWER(TRIM(p_env))
  END;

  -- resolve metadata
  SET v_target_catalog = (
    SELECT COALESCE(NULLIF(TRIM(cfg_target.target_catalog), ''), '{control_catalog}')
    FROM cfg
    LIMIT 1
  );

  SET v_target_schema = (
    SELECT NULLIF(TRIM(cfg_target.target_schema), '')
    FROM cfg
    LIMIT 1
  );

  SET v_target_object = (
    SELECT NULLIF(TRIM(cfg_target.target_object), '')
    FROM cfg
    LIMIT 1
  );

  SET v_target_format = (
    SELECT UPPER(COALESCE(NULLIF(TRIM(cfg_target.target_format), ''), 'DELTA'))
    FROM cfg
    LIMIT 1
  );

  SET v_domain = (
    SELECT LOWER(NULLIF(TRIM(cfg_config.domain), ''))
    FROM cfg
    LIMIT 1
  );

  -- treat catalog suffix by env
  SET v_target_catalog = CASE
    WHEN v_target_catalog RLIKE '^(?i).+_(dev|hml|prd)$' THEN v_target_catalog
    ELSE CONCAT(v_target_catalog, '_', v_env)
  END;

  -- build schema ddl
  SET v_schema_ddl = (
    SELECT ARRAY_JOIN(
      TRANSFORM(
        MAP_ENTRIES(cfg_target.target_schema_definition),
        col -> CONCAT('`', col.key, '` ', col.value)
      ),
      ', '
    )
    FROM cfg
    LIMIT 1
  );

  -- validations
  IF v_target_schema IS NULL THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_prepare_target: target_schema is null';
  END IF;

  IF v_target_object IS NULL THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_prepare_target: target_object is null';
  END IF;

  IF v_target_format <> 'DELTA' THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_prepare_target: only DELTA format is supported in v1';
  END IF;

  IF v_schema_ddl IS NULL OR TRIM(v_schema_ddl) = '' THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_prepare_target: target_schema_definition is null or empty';
  END IF;

  IF v_domain IS NULL THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_prepare_target: domain is required';
  END IF;

  IF v_env IS NULL OR TRIM(v_env) = '' THEN
    SIGNAL SQLSTATE '45000'
    SET MESSAGE_TEXT = 'proc_prepare_target: p_env is required';
  END IF;

  -- build external location
  SET v_target_location =
    'abfss://' || LOWER(TRIM(p_layer)) ||
    '@stdatacloud' || v_env || 'dblhbrs001.dfs.core.windows.net/' ||
    v_domain || '/' ||
    LOWER(TRIM(p_system)) || '/' ||
    LOWER(v_target_object);

  -- create schema
  SET v_create_schema =
    'CREATE SCHEMA IF NOT EXISTS `' || v_target_catalog || '`.`' || v_target_schema || '`';

  EXECUTE IMMEDIATE v_create_schema;

  -- check existence
  SET v_target_exists = (
    SELECT COUNT(*)
    FROM system.information_schema.tables
    WHERE LOWER(table_catalog) = LOWER(v_target_catalog)
      AND LOWER(table_schema) = LOWER(v_target_schema)
      AND LOWER(table_name) = LOWER(v_target_object)
  );

  -- create external table
  IF v_target_exists = 0 THEN
    SET v_create_table =
      'CREATE TABLE `' || v_target_catalog || '`.`' || v_target_schema || '`.`' || v_target_object || '` (' ||
      v_schema_ddl ||
      ") USING DELTA LOCATION '" || v_target_location || "'";

    EXECUTE IMMEDIATE v_create_table;
  END IF;
END;
""")